In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("bank.csv")
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,0,1,1,0,2343,1,0,2,5,8,1042,1,-1,0,3,1
1,56,0,1,1,0,45,0,0,2,5,8,1467,1,-1,0,3,1
2,41,9,1,1,0,1270,1,0,2,5,8,1389,1,-1,0,3,1
3,55,7,1,1,0,2476,1,0,2,5,8,579,1,-1,0,3,1
4,54,0,1,2,0,184,0,0,2,5,8,673,2,-1,0,3,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        11162 non-null  int64
 1   job        11162 non-null  int64
 2   marital    11162 non-null  int64
 3   education  11162 non-null  int64
 4   default    11162 non-null  int64
 5   balance    11162 non-null  int64
 6   housing    11162 non-null  int64
 7   loan       11162 non-null  int64
 8   contact    11162 non-null  int64
 9   day        11162 non-null  int64
 10  month      11162 non-null  int64
 11  duration   11162 non-null  int64
 12  campaign   11162 non-null  int64
 13  pdays      11162 non-null  int64
 14  previous   11162 non-null  int64
 15  poutcome   11162 non-null  int64
 16  deposit    11162 non-null  int64
dtypes: int64(17)
memory usage: 1.4 MB


In [4]:
obj_cols=df.drop("age",axis=1).columns
for i in obj_cols:
    print(f"{i} : {df[i].nunique()}\n{df[i].value_counts()}")


job : 12
job
4     2566
1     1944
9     1823
0     1334
7      923
5      778
6      405
8      360
10     357
2      328
3      274
11      70
Name: count, dtype: int64
marital : 3
marital
1    6351
2    3518
0    1293
Name: count, dtype: int64
education : 4
education
1    5476
2    3689
0    1500
3     497
Name: count, dtype: int64
default : 2
default
0    10994
1      168
Name: count, dtype: int64
balance : 3805
balance
 0       774
 1        39
 3        35
 2        34
 4        29
        ... 
 8585      1
-159       1
-132       1
 4576      1
 6691      1
Name: count, Length: 3805, dtype: int64
housing : 2
housing
0    5881
1    5281
Name: count, dtype: int64
loan : 2
loan
0    9702
1    1460
Name: count, dtype: int64
contact : 3
contact
0    8042
2    2346
1     774
Name: count, dtype: int64
day : 31
day
20    570
18    548
30    478
5     477
15    466
14    463
13    453
21    452
6     447
12    445
8     419
17    411
28    410
4     402
29    388
19    384
7     382
11  

In [5]:
features=df.drop("deposit",axis=1)
target = df["deposit"]

In [6]:
from sklearn.preprocessing import StandardScaler
ss=StandardScaler()
features.iloc[:]=ss.fit_transform(features.iloc[:])
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(features,target,random_state=1,test_size=0.2,stratify=target)

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [11]:
def mymodel(model):
    model.fit(xtrain,ytrain)
    ypred=model.predict(xtest)
    #Overfitting - underfitting check
    print(f"Training score : {model.score(xtrain,ytrain)}")
    print(f"Testing score : {model.score(xtest,ytest)}")
    c=pd.DataFrame(confusion_matrix(ypred,ytest),
                  index=["Not Subscribed","Subscribed"],
                  columns=["Not Subscribed","Subscribed"])
    print("Confusion Matrix : \n",c)
    print("Classification Report : \n",classification_report(ytest,ypred))
    return model

In [12]:
dt=mymodel(DecisionTreeClassifier())

Training score : 1.0
Testing score : 0.755038065382893
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             915         287
Subscribed                 260         771
Classification Report : 
               precision    recall  f1-score   support

           0       0.76      0.78      0.77      1175
           1       0.75      0.73      0.74      1058

    accuracy                           0.76      2233
   macro avg       0.75      0.75      0.75      2233
weighted avg       0.75      0.76      0.75      2233



In [13]:
ad=mymodel(AdaBoostClassifier())

Training score : 0.817784746332176
Testing score : 0.7984773846842812
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             958         233
Subscribed                 217         825
Classification Report : 
               precision    recall  f1-score   support

           0       0.80      0.82      0.81      1175
           1       0.79      0.78      0.79      1058

    accuracy                           0.80      2233
   macro avg       0.80      0.80      0.80      2233
weighted avg       0.80      0.80      0.80      2233



In [14]:
base_tree=DecisionTreeClassifier(max_depth=1)
mymodel(AdaBoostClassifier(estimator=base_tree))

Training score : 0.817784746332176
Testing score : 0.7984773846842812
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             958         233
Subscribed                 217         825
Classification Report : 
               precision    recall  f1-score   support

           0       0.80      0.82      0.81      1175
           1       0.79      0.78      0.79      1058

    accuracy                           0.80      2233
   macro avg       0.80      0.80      0.80      2233
weighted avg       0.80      0.80      0.80      2233



AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1))

In [18]:
parameters={
    "n_estimators":[80,120,200],
    "learning_rate":[0.1,0.001,0.01]
  
}

In [19]:
from sklearn.model_selection import GridSearchCV
clf=GridSearchCV(AdaBoostClassifier(),parameters,verbose=2)
clf.fit(xtrain,ytrain)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END .................learning_rate=0.1, n_estimators=80; total time=   0.5s
[CV] END .................learning_rate=0.1, n_estimators=80; total time=   0.4s
[CV] END .................learning_rate=0.1, n_estimators=80; total time=   0.4s
[CV] END .................learning_rate=0.1, n_estimators=80; total time=   0.4s
[CV] END .................learning_rate=0.1, n_estimators=80; total time=   0.4s
[CV] END ................learning_rate=0.1, n_estimators=120; total time=   0.8s
[CV] END ................learning_rate=0.1, n_estimators=120; total time=   0.7s
[CV] END ................learning_rate=0.1, n_estimators=120; total time=   0.7s
[CV] END ................learning_rate=0.1, n_estimators=120; total time=   0.7s
[CV] END ................learning_rate=0.1, n_estimators=120; total time=   0.7s
[CV] END ................learning_rate=0.1, n_estimators=200; total time=   1.1s
[CV] END ................learning_rate=0.1, n_est

GridSearchCV(estimator=AdaBoostClassifier(),
             param_grid={'learning_rate': [0.1, 0.001, 0.01],
                         'n_estimators': [80, 120, 200]},
             verbose=2)

In [20]:
mymodel(clf.best_estimator_)

Training score : 0.8061373054093404
Testing score : 0.7832512315270936
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             954         263
Subscribed                 221         795
Classification Report : 
               precision    recall  f1-score   support

           0       0.78      0.81      0.80      1175
           1       0.78      0.75      0.77      1058

    accuracy                           0.78      2233
   macro avg       0.78      0.78      0.78      2233
weighted avg       0.78      0.78      0.78      2233



AdaBoostClassifier(learning_rate=0.1, n_estimators=200)

In [21]:
from sklearn.ensemble import GradientBoostingClassifier

In [23]:
gb=mymodel(GradientBoostingClassifier())

Training score : 0.8593347519319072
Testing score : 0.8347514554411106
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             965         159
Subscribed                 210         899
Classification Report : 
               precision    recall  f1-score   support

           0       0.86      0.82      0.84      1175
           1       0.81      0.85      0.83      1058

    accuracy                           0.83      2233
   macro avg       0.83      0.84      0.83      2233
weighted avg       0.84      0.83      0.83      2233



In [24]:
parameters={
    "n_estimators":[70,110],
    "learning_rate":[0.01,0.1,0.05],
    "max_depth":[3,4,5,6]    
}

In [25]:
clf=GridSearchCV(GradientBoostingClassifier(),parameters,verbose=2)
clf.fit(xtrain,ytrain)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.8s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.7s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.8s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.7s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.7s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   1.1s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   1.2s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   1.3s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   1.1s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   1.1s
[CV] END ...learning_rate=0.01, max_depth=4, n_estimators=70; total time=   1.0s
[CV] END ...learning_rate=0.01, max_depth=4, n_

GridSearchCV(estimator=GradientBoostingClassifier(),
             param_grid={'learning_rate': [0.01, 0.1, 0.05],
                         'max_depth': [3, 4, 5, 6], 'n_estimators': [70, 110]},
             verbose=2)

In [26]:
gbc=mymodel(clf.best_estimator_)

Training score : 0.9241796393773098
Testing score : 0.8414688759516346
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             973         152
Subscribed                 202         906
Classification Report : 
               precision    recall  f1-score   support

           0       0.86      0.83      0.85      1175
           1       0.82      0.86      0.84      1058

    accuracy                           0.84      2233
   macro avg       0.84      0.84      0.84      2233
weighted avg       0.84      0.84      0.84      2233



In [ ]:
#XGBoost

In [27]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.4/72.0 MB 12.0 MB/s eta 0:00:06
   -- ------------------------------------- 5.0/72.0 MB 11.9 MB/s eta 0:00:06
   ---- ----------------------------------- 7.6/72.0 MB 11.9 MB/s eta 0:00:06
   ----- ---------------------------------- 10.0/72.0 MB 11.9 MB/s eta 0:00:06
   ------ --------------------------------- 12.6/72.0 MB 11.9 MB/s eta 0:00:06
   -------- ------------------------------- 15.2/72.0 MB 11.9 MB/s eta 0:00:05
   --------- ------------------------------ 17.6/72.0 MB 11.9 MB/s eta 0:00:05
   ----------- ---------------------------- 20.2/72.0 MB 11.9 MB/s eta 0:00:05
   ------------ --------------------------- 22.8/72.0 MB 11.8 MB/s eta 0:00:05
   -------------- ------------------------- 25.4/72.0 MB 11.9 MB/s eta 0:00:04
   --------------- ------------------------ 27.8/72.0 MB 11.8 MB/

In [28]:
from xgboost import XGBClassifier
xgb=mymodel(XGBClassifier())

Training score : 0.9569940642849143
Testing score : 0.8454993282579489
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             975         145
Subscribed                 200         913
Classification Report : 
               precision    recall  f1-score   support

           0       0.87      0.83      0.85      1175
           1       0.82      0.86      0.84      1058

    accuracy                           0.85      2233
   macro avg       0.85      0.85      0.85      2233
weighted avg       0.85      0.85      0.85      2233



In [29]:
clf=GridSearchCV(XGBClassifier(),parameters,verbose=2)
clf.fit(xtrain,ytrain)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=3, n_estimators=70; total time=   0.0s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   0.0s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   0.0s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   0.0s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   0.0s
[CV] END ..learning_rate=0.01, max_depth=3, n_estimators=110; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=4, n_estimators=70; total time=   0.0s
[CV] END ...learning_rate=0.01, max_depth=4, n_

GridSearchCV(estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     lea..., max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.01, 0.1, 0.05],
                         'max_depth': [3, 4, 5, 6], 'n_estimators': [70, 110]},
             verbose=2)

In [30]:
xgbc=mymodel(clf.best_estimator_)

Training score : 0.9048045693806698
Testing score : 0.8459471562919839
Confusion Matrix : 
                 Not Subscribed  Subscribed
Not Subscribed             974         143
Subscribed                 201         915
Classification Report : 
               precision    recall  f1-score   support

           0       0.87      0.83      0.85      1175
           1       0.82      0.86      0.84      1058

    accuracy                           0.85      2233
   macro avg       0.85      0.85      0.85      2233
weighted avg       0.85      0.85      0.85      2233

